# 06 — Build Analysis Panel

This notebook merges the three cleaned datasets into a single analysis-ready panel for the
border-county difference-in-differences design (Dube, Lester & Reich 2010).

**Inputs:**
- `data/intermediate/qcew_panel.parquet` — county × year × industry employment/wage data
- `data/intermediate/min_wage_panel.parquet` — state × year minimum wages
- `data/intermediate/border_pairs.parquet` — 1,055 cross-state county pairs with min wage divergence

**Output:** `data/processed/analysis_panel.parquet`
- Unit of observation: county × year × industry × pair_id

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path().resolve().parent
INTERMEDIATE = ROOT / "data" / "intermediate"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)
OUTPUT = PROCESSED / "analysis_panel.parquet"

QCEW_FILE = INTERMEDIATE / "qcew_panel.parquet"
MIN_WAGE_FILE = INTERMEDIATE / "min_wage_panel.parquet"
PAIRS_FILE = INTERMEDIATE / "border_pairs.parquet"

FIRST_YEAR = 2004
LAST_YEAR = 2024

## 1. Load inputs

In [ ]:
qcew = pd.read_parquet(QCEW_FILE)
mw = pd.read_parquet(MIN_WAGE_FILE)
pairs = pd.read_parquet(PAIRS_FILE)

print(f"QCEW panel     : {qcew.shape}")
print(f"Min wage panel : {mw.shape}")
print(f"Border pairs   : {pairs.shape}")
print()
print("QCEW columns  :", qcew.columns.tolist())
print("MW columns    :", mw.columns.tolist())
print("Pairs columns :", pairs.columns.tolist())

## 2. Impute missing state-years in the min wage panel

In [ ]:
FEDERAL_MIN = 7.25

# Build a complete state × year grid for (FIRST_YEAR-1)–LAST_YEAR
all_states = mw["state"].unique()
all_years = range(FIRST_YEAR - 1, LAST_YEAR + 1)
full_grid = pd.MultiIndex.from_product([all_states, all_years], names=["state", "year"])
full_grid = pd.DataFrame(index=full_grid).reset_index()

mw_complete = full_grid.merge(mw, on=["state", "year"], how="left")

# Identify missing state-years
missing = mw_complete[mw_complete["min_wage"].isna()]
print(f"Missing state-years before imputation: {len(missing)}")
print(missing[["state", "year"]])

In [ ]:
# Drop invalid state-years instead of imputing
# AZ and FL had no state minimum wage law for these years
# GA and WY 2024 have statutory rates below the federal floor and FRED has no value
drop_conditions = [
    ("AZ", [2003, 2004, 2005, 2006]),
    ("FL", [2003, 2004, 2005]),
    ("GA", [2024]),
    ("WY", [2024]),
]

for state, years in drop_conditions:
    mw_complete = mw_complete[
        ~((mw_complete["state"] == state) & (mw_complete["year"].isin(years)))
    ]

In [ ]:
# Impute state_fips for the missing rows
fips_lookup = mw.drop_duplicates("state").set_index("state")["state_fips"]
mw_complete["state_fips"] = mw_complete["state_fips"].fillna(
    mw_complete["state"].map(fips_lookup)
)

# Impute min_wage as federal floor
mw_complete["min_wage"] = mw_complete["min_wage"].fillna(FEDERAL_MIN)

print(
    f"\nMissing state-years after imputation : {mw_complete['min_wage'].isna().sum()}"
)
print(f"Total rows in completed panel        : {len(mw_complete):,}")

## 3. Filter QCEW to border counties and analysis years

In [ ]:
# All county FIPS that appear in at least one border pair
border_fips = set(pairs["fips1"]) | set(pairs["fips2"])
print(f"Unique border county FIPS : {len(border_fips):,}")

# Filter QCEW
qcew_border = qcew[
    qcew["area_fips"].isin(border_fips) & qcew["year"].between(FIRST_YEAR, LAST_YEAR)
].copy()

print(f"QCEW rows (all counties)  : {len(qcew):,}")
print(f"QCEW rows (border only)   : {len(qcew_border):,}")
print(f"Unique counties retained  : {qcew_border['area_fips'].nunique():,}")
print(f"Industries                : {qcew_border['industry_code'].unique().tolist()}")

## 4. Merge minimum wages onto QCEW

Derive each county's state from the first 2 digits of its FIPS code, map to state abbreviation
using the Census FIPS table, then merge the annual minimum wage.

In [ ]:
# Load FIPS → state abbreviation lookup
state_fips_df = pd.read_csv(
    ROOT / "data" / "raw" / "state_fips.txt",
    sep="|",
    dtype=str,
)[["STATE", "STUSAB"]].rename(columns={"STATE": "state_fips", "STUSAB": "state"})
state_fips_df["state_fips"] = state_fips_df["state_fips"].str.zfill(2)
fips_to_abbr = dict(zip(state_fips_df["state_fips"], state_fips_df["state"]))

# Derive state from county FIPS
qcew_border["state_fips"] = qcew_border["area_fips"].str[:2]
qcew_border["state"] = qcew_border["state_fips"].map(fips_to_abbr)

# Merge minimum wage
mw_analysis = mw_complete[mw_complete["year"].between(FIRST_YEAR, LAST_YEAR)]

qcew_mw = qcew_border.merge(
    mw_analysis[["state", "year", "min_wage"]],
    on=["state", "year"],
    how="left",
)

unmatched_mw = qcew_mw["min_wage"].isna().sum()
print(f"Rows after merging min wages : {len(qcew_mw):,}")
print(f"Rows with no min wage match  : {unmatched_mw:,}  (should be 0)")
qcew_mw[["area_fips", "state", "year", "industry_code", "min_wage"]].head()

## 5. Attach pair IDs

Each border county may appear in multiple pairs (e.g. a corner county bordered by two different
states). We keep the panel long — one row per county × year × industry × pair.

In [ ]:
# Explode pairs into two long tables — one per side of the pair
pairs_left = pairs[["pair_id", "fips1", "fips2", "state1", "state2"]].rename(
    columns={
        "fips1": "area_fips",
        "state1": "pair_own_state",
        "state2": "pair_other_state",
    }
)
pairs_right = pairs[["pair_id", "fips1", "fips2", "state1", "state2"]].rename(
    columns={
        "fips2": "area_fips",
        "state2": "pair_own_state",
        "state1": "pair_other_state",
    }
)

# Stack and keep relevant columns
pairs_long = pd.concat([pairs_left, pairs_right], ignore_index=True)[
    ["pair_id", "area_fips", "pair_own_state", "pair_other_state"]
]

# Merge onto QCEW
panel = qcew_mw.merge(pairs_long, on="area_fips", how="inner")

print(f"Rows before pair merge : {len(qcew_mw):,}")
print(
    f"Rows after pair merge  : {len(panel):,}  (more rows = counties in multiple pairs)"
)
print(f"Unique pair_ids        : {panel['pair_id'].nunique():,}")
print(f"Unique counties        : {panel['area_fips'].nunique():,}")

## 6. Create log-transformed outcome variables

Following Dube et al. (2010), we work in logs throughout:
- `log_emp` — log average employment (primary outcome)
- `log_wage` — log average weekly wage
- `log_estabs` — log establishment count
- `log_min_wage` — log minimum wage (the treatment variable)

We add 1 inside the log for employment and establishments to handle any zero counts without
dropping rows.

In [ ]:
# Ensure numeric types
for col in [
    "annual_avg_emplvl",
    "annual_avg_wkly_wage",
    "annual_avg_estabs_count",
    "min_wage",
]:
    panel[col] = pd.to_numeric(panel[col], errors="coerce")

# Log transformations
panel["log_emp"] = np.log(panel["annual_avg_emplvl"] + 1)
panel["log_wage"] = np.log(panel["annual_avg_wkly_wage"].replace(0, np.nan))
panel["log_estabs"] = np.log(panel["annual_avg_estabs_count"] + 1)
panel["log_min_wage"] = np.log(panel["min_wage"])

print("Log variable summary:")
print(panel[["log_emp", "log_wage", "log_estabs", "log_min_wage"]].describe().round(3))

## 7. Flag treated county in each pair × year

Within each pair and year, the county in the **higher minimum wage state** is `treated = 1`.
When both states have the same wage in a given year, `treated = 0` for both (no variation that year).

In [ ]:
# Merge the partner state's minimum wage for comparison
partner_mw = mw_analysis[["state", "year", "min_wage"]].rename(
    columns={"state": "pair_other_state", "min_wage": "partner_min_wage"}
)
panel = panel.merge(partner_mw, on=["pair_other_state", "year"], how="left")

# treated = 1 if this county's state has a strictly higher minimum wage than its partner
panel["treated"] = (panel["min_wage"] > panel["partner_min_wage"]).astype(int)

print("Treated distribution across pair × year × county × industry rows:")
print(panel["treated"].value_counts())
print(f"\nTreatment rate: {panel['treated'].mean():.1%}")

## 8. Drop BLS-suppressed rows

In [ ]:
n_before = len(panel)
suppressed = (panel["disclosure_code"] == "N").sum()
print(f"Rows before suppression drop : {n_before:,}")
print(f"Suppressed rows (code = 'N') : {suppressed:,}  ({suppressed/n_before:.1%})")

panel = panel[panel["disclosure_code"] != "N"].copy()
print(f"Rows after suppression drop  : {len(panel):,}")

## 9. Final panel — select and order columns

In [ ]:
FINAL_COLS = [
    # Identifiers
    "pair_id",
    "area_fips",
    "state",
    "year",
    "industry_code",
    "industry_title",
    # Treatment
    "treated",
    "min_wage",
    "partner_min_wage",
    "log_min_wage",
    # Outcomes (raw)
    "annual_avg_emplvl",
    "annual_avg_wkly_wage",
    "annual_avg_estabs_count",
    "total_annual_wages",
    "avg_annual_pay",
    # Outcomes (log)
    "log_emp",
    "log_wage",
    "log_estabs",
    # Metadata
    "area_title",
]

panel_final = (
    panel[FINAL_COLS]
    .sort_values(["pair_id", "area_fips", "industry_code", "year"])
    .reset_index(drop=True)
)

print("Final panel shape:", panel_final.shape)
print("\nMissing values:")
print(panel_final.isnull().sum()[panel_final.isnull().sum() > 0])
panel_final.head(10)

## 10. Summary statistics

In [ ]:
print("=" * 55)
print("PANEL SUMMARY")
print("=" * 55)
print(f"Total rows                   : {len(panel_final):,}")
print(f"Unique pair_ids              : {panel_final['pair_id'].nunique():,}")
print(f"Unique counties              : {panel_final['area_fips'].nunique():,}")
print(f"Unique states                : {panel_final['state'].nunique()}")
print(f"Years                        : {FIRST_YEAR}–{LAST_YEAR}")
print(f"Industries                   : {panel_final['industry_code'].nunique()}")
print(f"Treatment rate               : {panel_final['treated'].mean():.1%}")
print()
print("Rows by industry:")
print(
    panel_final.groupby(["industry_code", "industry_title"])
    .size()
    .reset_index(name="n_rows")
    .to_string(index=False)
)
print()
print("Mean log employment by industry:")
print(panel_final.groupby("industry_code")["log_emp"].mean().round(3).to_string())

## 11. Save to parquet

In [ ]:
panel_final.to_parquet(OUTPUT, index=False)